In [14]:
import pandas as pd
import numpy as np

EPS = 1e-3  # sMAPE 안정화

def load_any(path):
    # utf-8 / cp949 호환 로더
    for enc in ["utf-8", "cp949"]:
        try: return pd.read_csv(path, encoding=enc)
        except: pass
    return pd.read_csv(path)

def to_long(df):
    cols = df.columns
    if ("영업일자" in cols) and ("영업장명_메뉴명" in cols) and ("매출수량" in cols):
        return df[["영업일자","영업장명_메뉴명","매출수량"]].copy()
    # wide → long
    date_col = next((c for c in cols if c=="영업일자" or "일자" in c or "date" in c.lower()), None)
    if date_col is None:
        df = df.copy(); df.insert(0, "영업일자", np.arange(len(df))); date_col = "영업일자"
    long = df.melt(id_vars=[date_col], var_name="영업장명_메뉴명", value_name="매출수량")
    long = long.rename(columns={date_col:"영업일자"})
    return long

def smape_pair(a, b, eps=EPS):
    num = (a - b).abs()
    den = (a.abs() + b.abs()).clip(lower=eps)
    return (2.0 * num / den)

# --- 경로만 바꿔서 실행 ---
path_attn = "./ensemble_submission_p90.csv"  # 새 모델(Attention)
path_base = "./ensemble_submission_1_clipped.csv"   # 기존 v7 or v8

p1 = to_long(load_any(path_attn)); p1 = p1.rename(columns={"매출수량":"pred_attn"})
p2 = to_long(load_any(path_base)); p2 = p2.rename(columns={"매출수량":"pred_base"})

m = pd.merge(p1, p2, on=["영업일자","영업장명_메뉴명"], how="inner")
m["smape_pair"] = smape_pair(m["pred_attn"], m["pred_base"])

print("Pairwise sMAPE (mean) =", m["smape_pair"].mean().round(4))
print("Pairwise sMAPE (p90)  =", m["smape_pair"].quantile(0.90).round(4))

# 어디서 차이가 큰지 TOP 리스트
top_rows = m.nlargest(30, "smape_pair")[["영업일자","영업장명_메뉴명","pred_attn","pred_base","smape_pair"]]
top_items = (m.groupby("영업장명_메뉴명")["smape_pair"].mean()
               .sort_values(ascending=False).head(30).reset_index())
display(top_rows); display(top_items)


Pairwise sMAPE (mean) = 0.0103
Pairwise sMAPE (p90)  = 0.0


,영업일자,영업장명_메뉴명,pred_attn,pred_base,smape_pair
11075,TEST_02+2일,카페테리아_오픈푸드,29.0,59.000000,0.681818
11076,TEST_02+3일,카페테리아_오픈푸드,29.0,59.000000,0.681818
11077,TEST_02+4일,카페테리아_오픈푸드,29.0,59.000000,0.681818
11118,TEST_08+3일,카페테리아_오픈푸드,29.0,59.000000,0.681818
11119,TEST_08+4일,카페테리아_오픈푸드,29.0,59.000000,0.681818
11120,TEST_08+5일,카페테리아_오픈푸드,29.0,59.000000,0.681818
11126,TEST_09+4일,카페테리아_오픈푸드,29.0,59.000000,0.681818
11127,TEST_09+5일,카페테리아_오픈푸드,29.0,59.000000,0.681818
636,TEST_00+7일,느티나무 셀프BBQ_쌈장,1.0,2.000000,0.666667
4832,TEST_00+3일,라그로타_Open Food,1.0,2.000000,0.666667


,영업장명_메뉴명,smape_pair
0,미라시아_브런치 4인 패키지,0.128232
1,카페테리아_오픈푸드,0.127684
2,담하_(단체) 공깃밥,0.120181
3,연회장_야채추가,0.099769
4,라그로타_Open Food,0.091951
5,라그로타_모둠 해산물 플래터,0.089430
6,라그로타_한우 (200g),0.069990
7,라그로타_양갈비 (4ps),0.068759
8,미라시아_잭 애플 토닉,0.063498
9,담하_더덕 한우 지짐,0.059649
